In [23]:
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import Trainer, TrainingArguments
from transformers import pipeline
import torch
import pandas as pd
from sklearn.model_selection import train_test_split

In [24]:
reviews_df = pd.read_csv("data/output/processed/reviews.csv")

print(reviews_df.head())
print(reviews_df.columns)

  review_id meal_id            meal_name      restaurant  rating  \
0   R000001   M0001  Classic Beef Burger  Mama's Kitchen     1.9   
1   R000002   M0001  Classic Beef Burger  Mama's Kitchen     3.8   
2   R000003   M0001  Classic Beef Burger  Mama's Kitchen     4.0   
3   R000004   M0001  Classic Beef Burger  Mama's Kitchen     4.7   
4   R000005   M0001  Classic Beef Burger  Mama's Kitchen     4.0   

                                         review_text sentiment_label  \
0  Disappointed with the quality. Not what I expe...        negative   
1  Pretty good, I've had better but wouldn't say ...         neutral   
2  Super fast delivery and the food was outstandi...        positive   
3  Perfect seasoning and amazing presentation. Wo...        positive   
4  Loved every bite. The packaging was also excel...        positive   

  review_date  helpful_votes  
0  2026-03-19              8  
1  2026-04-05             44  
2  2026-02-01             50  
3  2025-12-14             10  
4  

In [25]:
sentiment_pipe = pipeline("sentiment-analysis")

def get_label(text):
    if pd.isna(text):
        return 1  # neutral
    
    result = sentiment_pipe(str(text)[:512])[0]['label']
    
    if result == "NEGATIVE":
        return 0
    else:
        return 2  # positive

reviews_df['label'] = reviews_df['review_text'].apply(get_label)

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [26]:
train_texts, test_texts, train_labels, test_labels = train_test_split(
    reviews_df['review_text'].tolist(),
    reviews_df['label'].tolist(),
    test_size=0.2,
    random_state=42
)

In [27]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

train_encodings = tokenizer(
    train_texts,
    truncation=True,
    padding=True,
    max_length=128
)

test_encodings = tokenizer(
    test_texts,
    truncation=True,
    padding=True,
    max_length=128
)

In [28]:
class ReviewsDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = ReviewsDataset(train_encodings, train_labels)
test_dataset = ReviewsDataset(test_encodings, test_labels)

In [29]:
model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=3
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [30]:
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_dir='./logs',
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [31]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

In [32]:
trainer.train()

c:\Users\HP\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=368, training_loss=0.0288338013317274, metrics={'train_runtime': 735.7827, 'train_samples_per_second': 3.99, 'train_steps_per_second': 0.5, 'total_flos': 34702193110032.0, 'train_loss': 0.0288338013317274, 'epoch': 2.0})

In [33]:
trainer.evaluate()

c:\Users\HP\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Step
No log,0.000294,368


{'eval_loss': 0.0002940235426649451}

In [34]:
def predict_sentiment(text):
    inputs = tokenizer(
        str(text),
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    outputs = model(**inputs)
    pred = torch.argmax(outputs.logits, dim=1).item()

    if pred == 0:
        return "NEGATIVE", -1
    elif pred == 1:
        return "NEUTRAL", 0
    else:
        return "POSITIVE", 1

In [35]:
results = reviews_df['review_text'].apply(predict_sentiment)

reviews_df['sentiment'] = results.apply(lambda x: x[0])
reviews_df['sentiment_score'] = results.apply(lambda x: x[1])

In [37]:
reviews_df = reviews_df[
    ['review_id', 'meal_id', 'review_text', 'sentiment', 'sentiment_score']
]

reviews_df.to_csv("data/output/processed/reviews_with_sentiment.csv", index=False)